# Conformal Prediction for Directed Graph Link Prediction

This notebook presents a **directed link prediction** framework using **split conformal prediction** on the **Wikivitals** dataset. This produce prediction sets $C(x) \subseteq \{0, 1\}$ with guaranteed coverage at a chosen confidence level (e.g., 90%).

## Key Differences from Undirected Graphs

1. **Asymmetric adjacency matrix**: $A_{ij} \neq A_{ji}$ in general
2. **Directed embeddings**: Separate source and target embeddings
3. **In-degree and out-degree**: Two distinct degree metrics per node
4. **Directed features**: Asymmetric similarity and structural features

In [11]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
from sknetwork.embedding import SVD
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_fscore_support

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

## 1. Mathematical Foundations

### 1.1 Directed Graph Representation

A directed graph $G = (V, E)$ consists of:
- A set of vertices $V = \{v_1, v_2, \ldots, v_n\}$
- A set of directed edges $E \subseteq V \times V$, where $(u, v) \in E$ represents an edge from $u$ to $v$

The **adjacency matrix** $A \in \{0,1\}^{n \times n}$ is defined as:

$$
A_{ij} = \begin{cases}
1 & \text{if } (v_i \to v_j) \in E \\
0 & \text{otherwise}
\end{cases}
$$

Note: $A_{ij} = 1$ does NOT imply $A_{ji} = 1$ (asymmetry).

### 1.2 Node Degree Metrics

For a directed graph, each node $v_i$ has two degree measures:

**Out-degree** (number of outgoing edges):
$$
d^{\text{out}}_i = \sum_{j=1}^{n} A_{ij}
$$

**In-degree** (number of incoming edges):
$$
d^{\text{in}}_i = \sum_{j=1}^{n} A_{ji}
$$

### 1.3 Mondrian (Label-Conditional) Split Conformal Quantiles

For each class $y \in \{0, 1\}$, define the calibration index set and count:
$$\mathcal{I}_y = \{i \in \{1,\dots,n\} : y_i = y\}, \qquad n_y = |\mathcal{I}_y|$$

Compute the finite-sample rank:
$$k_y = \lceil (n_y + 1)(1 - \alpha) \rceil$$

The quantile $q_y$ is the $k_y$-th order statistic of the class-$y$ calibration scores:
$$q_y = s^{(y)}_{(k_y)} \quad \text{where} \quad s^{(y)}_{(1)} \le s^{(y)}_{(2)} \le \cdots \le s^{(y)}_{(n_y)}$$

**Non-conformity scores** (with $p_i = P(Y=1|x_i)$):
$$s(x, 1) = 1 - p(x), \qquad s(x, 0) = p(x)$$

**Class-conditional quantiles**:
$$q_1 = (1 - p_i)_{(k_1)}, \; i \in \mathcal{I}_1, \qquad q_0 = (p_i)_{(k_0)}, \; i \in \mathcal{I}_0$$

**Probability thresholds**:
$$t_{\text{upper}} = 1 - q_1, \qquad t_{\text{lower}} = q_0$$

If $k_y > n_y$, set $q_y = \infty$ to guarantee coverage.

This ensures marginal coverage: $\mathbb{P}(Y \in C(X)) \geq 1 - \alpha$.

In [12]:
def conformal_quantile(scores, alpha):
    """
    Mondrian (label-conditional) split conformal quantile.
    
    Returns the k-th order statistic where k = ceil((n+1)(1-alpha)).
    If k > n, returns infinity to guarantee coverage.
    
    Parameters:
    -----------
    scores : array-like
        Calibration non-conformity scores for a single class
    alpha : float
        Target error rate (e.g., 0.1 for 90% coverage)
    
    Returns:
    --------
    float : The conformal quantile threshold q_y
    """
    n = len(scores)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    
    # If k > n, return infinity to ensure coverage
    if k > n:
        return np.inf
    
    # Return the k-th order statistic (k-th smallest value)
    sorted_scores = np.sort(scores)
    return sorted_scores[k - 1]  # k-1 for 0-indexed array

print("Conformal quantile function ready.")

Conformal quantile function ready.


## 2. Directed Graph Data Loading

### 2.1 Directed Adjacency Matrix Construction

For a directed graph, we construct the adjacency matrix **without** symmetrization:

$$
A_{ij} = 1 \iff (i \to j) \in E
$$

Key constraints:
- No self-loops: $A_{ii} = 0$ for all $i$
- Asymmetric: $A_{ij}$ is independent of $A_{ji}$

The input file format is:
```
source_node    target_node    True
```

where each line represents a directed edge from `source_node` to `target_node`.

In [13]:
def load_directed_graph_sparse(path):
    """
    Load a directed graph from a TSV file.
    
    Parameters:
    -----------
    path : str
        Path to adjacency.tsv file with columns: source, target, True
    
    Returns:
    --------
    scipy.sparse.csr_matrix : Directed adjacency matrix A where A[i,j] = 1 means i -> j
    """
    df = pd.read_csv(path, sep='\t', header=None, usecols=[0,1], names=['source', 'target'])
    
    # For directed graphs, we do NOT add reverse edges
    rows = df['source'].values
    cols = df['target'].values
    data = np.ones(len(rows))
    
    n_nodes = max(rows.max(), cols.max()) + 1
    
    # Create directed adjacency matrix
    adj = sp.csr_matrix((data, (rows, cols)), shape=(n_nodes, n_nodes))
    
    # Remove duplicates and self-loops
    adj.data = np.ones_like(adj.data)
    adj.setdiag(0)
    adj.eliminate_zeros()
    
    return adj

print("Directed graph loader ready.")

Directed graph loader ready.


## 3. Directed Graph Embeddings

### 3.1 Spectral Embeddings via SVD

For a directed adjacency matrix $A \in \mathbb{R}^{n \times n}$, we apply **Singular Value Decomposition**:

$$
A \approx U \Sigma V^T
$$

where:
- $U \in \mathbb{R}^{n \times k}$: **Source node embeddings** (captures outgoing edge patterns)
- $V \in \mathbb{R}^{n \times k}$: **Target node embeddings** (captures incoming edge patterns)
- $\Sigma \in \mathbb{R}^{k \times k}$: Singular values (diagonal)
- $k$: Embedding dimension (e.g., 32)

This gives us:
- **Source embedding** for node $i$: $u_i \in \mathbb{R}^k$ (row $i$ of $U$)
- **Target embedding** for node $j$: $v_j \in \mathbb{R}^k$ (row $j$ of $V$)

The reconstruction approximates: $A_{ij} \approx u_i^T \Sigma v_j$

### 3.2 Why Two Embeddings?

In directed graphs:
- A node's **role as a source** (outgoing edges) differs from its **role as a target** (incoming edges)
- Example: In a citation network, a paper cited by many (high in-degree) may cite few (low out-degree)
- Separate embeddings capture these asymmetric roles

In [14]:
def get_directed_node_features(adj, n_components=32):
    """
    Extract node features for directed graphs.
    
    Parameters:
    -----------
    adj : scipy.sparse matrix
        Directed adjacency matrix
    n_components : int
        Embedding dimension
    
    Returns:
    --------
    out_degrees : array, shape (n_nodes,)
        Out-degree of each node
    in_degrees : array, shape (n_nodes,)
        In-degree of each node
    source_embedding : array, shape (n_nodes, n_components)
        Source (row) embeddings from SVD
    target_embedding : array, shape (n_nodes, n_components)
        Target (column) embeddings from SVD
    """
    # Out-degrees: sum along columns (axis=1)
    out_degrees = np.array(adj.sum(axis=1)).flatten()
    
    # In-degrees: sum along rows (axis=0)
    in_degrees = np.array(adj.sum(axis=0)).flatten()
    
    # SVD Embeddings
    print(f"Computing SVD Embeddings (Rank {n_components}) for directed graph...")
    svd = SVD(n_components=n_components)
    
    # For directed graphs, sknetwork SVD returns row embeddings (U) and col embeddings (V)
    # fit_transform returns row embeddings (source embeddings)
    source_embedding = svd.fit_transform(adj)
    
    # Get column embeddings V (target embeddings) using correct sknetwork attribute
    target_embedding = svd.embedding_col_
    
    return out_degrees, in_degrees, source_embedding, target_embedding

print("Directed feature extractor ready.")

Directed feature extractor ready.


## 4. Feature Engineering for Directed Edge Prediction

### 4.1 Directed Edge Features

For a candidate directed edge $(u \to v)$, we construct feature vector $x_{u \to v}$:

$$
x_{u \to v} = \begin{bmatrix}
d^{\text{out}}_u \\
d^{\text{in}}_v \\
d^{\text{in}}_u \\
d^{\text{out}}_v \\
\text{sim}(u, v) \\
\text{sim}_{\text{reverse}}(v, u)
\end{bmatrix} \in \mathbb{R}^6
$$

where:

**Degree features** (capture node importance):
- $d^{\text{out}}_u$: Out-degree of source node (how many nodes $u$ points to)
- $d^{\text{in}}_v$: In-degree of target node (how many nodes point to $v$)
- $d^{\text{in}}_u$: In-degree of source node
- $d^{\text{out}}_v$: Out-degree of target node

**Directed similarity** (captures embedding compatibility):

$$
\text{sim}(u, v) = \frac{u_{\text{source}} \cdot v_{\text{target}}}{\|u_{\text{source}}\| \|v_{\text{target}}\|}
$$

This measures how well $u$'s outgoing pattern matches $v$'s incoming pattern.

**Reverse similarity** (captures potential reciprocity):

$$
\text{sim}_{\text{reverse}}(v, u) = \frac{v_{\text{source}} \cdot u_{\text{target}}}{\|v_{\text{source}}\| \|u_{\text{target}}\|}
$$

This measures the likelihood of the reverse edge $v \to u$.

### 4.2 Intuition

- If $u$ has high out-degree and $v$ has high in-degree, edge $u \to v$ is more likely (hub pattern)
- High $\text{sim}(u, v)$ means $u$'s outgoing neighbors are similar to $v$'s incoming neighbors
- High $\text{sim}_{\text{reverse}}(v, u)$ suggests potential reciprocal relationship

In [15]:
def compute_directed_pair_features(pairs, out_degrees, in_degrees, source_emb, target_emb):
    """
    Compute features for directed edge pairs (u -> v).
    
    Parameters:
    -----------
    pairs : array, shape (n_pairs, 2)
        Each row is [source, target] representing edge source -> target
    out_degrees : array
        Out-degree of each node
    in_degrees : array
        In-degree of each node
    source_emb : array, shape (n_nodes, k)
        Source embeddings (U from SVD)
    target_emb : array, shape (n_nodes, k)
        Target embeddings (V from SVD)
    
    Returns:
    --------
    features : array, shape (n_pairs, 6)
        Feature matrix with columns:
        [out_deg_u, in_deg_v, in_deg_u, out_deg_v, sim_forward, sim_reverse]
    """
    u = pairs[:, 0]  # Source nodes
    v = pairs[:, 1]  # Target nodes
    
    # Degree features
    out_deg_u = out_degrees[u]
    in_deg_v = in_degrees[v]
    in_deg_u = in_degrees[u]
    out_deg_v = out_degrees[v]
    
    # Forward similarity: u_source · v_target
    vec_u_source = source_emb[u]
    vec_v_target = target_emb[v]
    
    norm_u_source = np.linalg.norm(vec_u_source, axis=1)
    norm_v_target = np.linalg.norm(vec_v_target, axis=1)
    norm_u_source[norm_u_source == 0] = 1e-9
    norm_v_target[norm_v_target == 0] = 1e-9
    
    dot_forward = np.sum(vec_u_source * vec_v_target, axis=1)
    sim_forward = dot_forward / (norm_u_source * norm_v_target)
    
    # Reverse similarity: v_source · u_target (for reciprocity)
    vec_v_source = source_emb[v]
    vec_u_target = target_emb[u]
    
    norm_v_source = np.linalg.norm(vec_v_source, axis=1)
    norm_u_target = np.linalg.norm(vec_u_target, axis=1)
    norm_v_source[norm_v_source == 0] = 1e-9
    norm_u_target[norm_u_target == 0] = 1e-9
    
    dot_reverse = np.sum(vec_v_source * vec_u_target, axis=1)
    sim_reverse = dot_reverse / (norm_v_source * norm_u_target)
    
    return np.column_stack([out_deg_u, in_deg_v, in_deg_u, out_deg_v, sim_forward, sim_reverse])

print("Directed pair feature extractor ready.")

Directed pair feature extractor ready.


## 5. OGB Static Transductive Link Prediction


The idea is from the OGB transductive link prediction protocol, where the representation graph is constructed using training edges only, and validation/test edges are held out and used only for evaluation

Hu et al., Open Graph Benchmark: Datasets for Machine Learning on Graphs (NeurIPS 2020)
https://arxiv.org/abs/2005.00687

https://ogb.stanford.edu/docs/linkprop/
### 5.1 Edge Splitting (Exchangeability)

We randomly shuffle all directed edges $E$ and split them into disjoint sets:
$$
E = E_{\text{cal}} \cup E_{\text{test}} \cup E_{\text{train}}
$$
with $10\%$ for calibration, $10\%$ for testing, and $80\%$ for training.

### 5.2 Disjoint Message vs Supervision Edges (No Leakage)

Following the OGB protocol, we further split training edges:
$$
E_{\text{train}} = E_{\text{msg}} \cup E_{\text{sup}}, \quad E_{\text{msg}} \cap E_{\text{sup}} = \emptyset
$$

| Split | Purpose | Default |
|-------|---------|---------|
| $E_{\text{msg}}$ | Build representation graph $G_{\text{rep}}$ for SVD embeddings | 70% of $E_{\text{train}}$ |
| $E_{\text{sup}}$ | Classifier supervision (training labels) | 30% of $E_{\text{train}}$ |

This prevents the subtle leakage where the classifier learns to exploit edges already encoded in the SVD embeddings.

### 5.3 Representation Graph Construction

For directed graphs:
$$
G_{\text{rep}} = (V, E_{\text{msg}}) \quad \text{(asymmetric)}
$$
All node features (in/out-degrees, source/target SVD embeddings) are computed exclusively from $G_{\text{rep}}$.

### 5.4 Realistic Negative Sampling (No Oracle Access)

**Key principle**: When sampling negatives, we only exclude edges we would legitimately know at that point in time. This prevents data leakage from using test/calibration positives to filter negatives.

Define **known edges** (training edges only):
$$
E_{\text{known}} = E_{\text{msg}} \cup E_{\text{sup}}
$$

Negative sampling per split:

| Split | Exclude Set | Rationale |
|-------|-------------|-----------|
| Train ($E_{\text{sup}}$) | $E_{\text{known}}$ | Only exclude training edges |
| Calibration ($E_{\text{cal}}$) | $E_{\text{known}} \cup E_{\text{cal}}$ | Exclude train + cal positives (we know cal labels when calibrating) |
| Test ($E_{\text{test}}$) | $E_{\text{known}}$ | **DO NOT** exclude $E_{\text{test}}$ positives (no oracle access) |

This means test negatives could potentially be calibration or test positives—matching real deployment where we don't have oracle knowledge of ground truth.

### 5.5 Training & Conformal Calibration

1. Train classifier on $(E_{\text{sup}}^+, E_{\text{sup}}^-)$ using features from $G_{\text{rep}}$
2. Calibrate conformal thresholds on $(E_{\text{cal}}^+, E_{\text{cal}}^-)$
3. Evaluate on $(E_{\text{test}}^+, E_{\text{test}}^-)$

### 5.6 Non-conformity Scores

For each calibration sample $(x_i, y_i)$:
$$s_i(y) = \begin{cases}
1 - \hat{f}(x_i) & \text{if } y = 1 \\
\hat{f}(x_i) & \text{if } y = 0
\end{cases}$$

### 5.7 Prediction Sets

$$
C(x_{\text{test}}) = \{y \in \{0,1\} : s(x, y) \le q_y\}
$$

### 5.8 Coverage Guarantee

Under exchangeability: $\mathbb{P}(Y_{\text{test}} \in C(X_{\text{test}})) \geq 1 - \alpha$

## 5.5 Per-Source Corruption Negative Sampling (HeaRT / OGB Style)

### Motivation

Traditional uniform random negative sampling has a critical flaw identified by the **HeaRT paper** (Galkin et al., 2024):

> *"Random negatives don't share endpoints with positives, allowing models to exploit node-level features rather than learning true edge plausibility."*

### The Problem with Uniform Negatives

With uniform sampling, for a positive edge $(u \to v)$:
- Negative $(u', v')$ is sampled where both $u'$ and $v'$ are random
- The model can "cheat" by learning that certain nodes are more likely to be sources/targets
- This inflates AUC scores and doesn't reflect real link prediction tasks

### Per-Source Corruption Solution

For each positive edge $(u \to v^+)$, we **corrupt the target** while keeping the source fixed:

$$
\text{Negatives for } (u \to v^+): \{(u \to v^-_1), (u \to v^-_2), \ldots, (u \to v^-_K)\}
$$

where each $v^-_k$ satisfies:
1. $v^-_k \neq u$ (no self-loops)
2. $v^-_k \neq v^+$ (not the positive target)
3. $(u \to v^-_k) \notin E_{\text{known}}$ (not a known edge)

### Why This Is Better

| Aspect | Uniform Sampling | Per-Source Corruption |
|--------|-----------------|----------------------|
| **Task** | "Is this random pair an edge?" | "Given source $u$, which target should it link to?" |
| **Negatives share endpoint** | No | Yes (same source) |
| **Realistic for recommendations** | No | Yes |
| **Matches OGB/HeaRT** | No | Yes |

### Ranking Evaluation with MRR and Hits@K

With per-source negatives, we can compute **ranking metrics**:

**Mean Reciprocal Rank (MRR)**:
$$
\text{MRR} = \frac{1}{|E_{\text{test}}^+|} \sum_{(u, v^+) \in E_{\text{test}}^+} \frac{1}{\text{rank}(v^+)}
$$

where $\text{rank}(v^+)$ is the position of the true target among $\{v^+, v^-_1, \ldots, v^-_K\}$ sorted by predicted probability.

**Hits@K**:
$$
\text{Hits@}K = \frac{1}{|E_{\text{test}}^+|} \sum_{(u, v^+)} \mathbb{1}[\text{rank}(v^+) \leq K]
$$

### Implementation Parameters

| Parameter | Description | Default |
|-----------|-------------|---------|
| `K` | Number of negative candidates per positive | 50 |
| `out_neighbors` | Set of known targets per source (for filtering) | From $E_{\text{known}}$ |

In [16]:
def build_out_neighbors(edges, n_nodes):
    """
    Build a dictionary mapping each source node to its set of known targets.
    
    Parameters:
    -----------
    edges : array, shape (n_edges, 2)
        Array of directed edges [source, target]
    n_nodes : int
        Total number of nodes in the graph
    
    Returns:
    --------
    out_neighbors : dict
        Dictionary where out_neighbors[u] = set of nodes that u links to
    """
    out_neighbors = {u: set() for u in range(n_nodes)}
    for (u, v) in edges:
        out_neighbors[u].add(v)
    return out_neighbors


def sample_negatives_corrupt_tail(pos_edges, K, rng, n_nodes, out_neighbors, forbidden_edges_set=None):
    """
    Sample K negative edges per positive by corrupting the target (HeaRT/OGB style).
    
    For each positive edge (u -> v+), sample K negatives (u -> v-) where:
    - v- != u (no self-loops)
    - v- != v+ (not the positive target)
    - v- not in out_neighbors[u] (not a known outgoing edge from u)
    - (u, v-) not in forbidden_edges_set (optional additional filtering)
    
    Parameters:
    -----------
    pos_edges : array, shape (n_pos, 2)
        Positive edges [source, target]
    K : int
        Number of negative candidates per positive edge
    rng : numpy.random.Generator
        Random number generator
    n_nodes : int
        Total number of nodes
    out_neighbors : dict
        Dictionary mapping source -> set of known targets
    forbidden_edges_set : set, optional
        Additional edges to exclude (e.g., calibration positives)
    
    Returns:
    --------
    neg_edges : array, shape (n_pos * K, 2)
        Negative edges, ordered so that neg_edges[i*K:(i+1)*K] are the K negatives
        for pos_edges[i]
    """
    if forbidden_edges_set is None:
        forbidden_edges_set = set()
    
    neg_edges = []
    
    for (u, v_pos) in pos_edges:
        u = int(u)
        v_pos = int(v_pos)
        k = 0
        attempts = 0
        max_attempts = K * 100  # Prevent infinite loops for high-degree nodes
        
        while k < K and attempts < max_attempts:
            v_neg = int(rng.integers(0, n_nodes))
            attempts += 1
            
            # Skip invalid candidates
            if v_neg == u:  # No self-loops
                continue
            if v_neg == v_pos:  # Not the positive target
                continue
            if v_neg in out_neighbors[u]:  # Not an existing edge from u
                continue
            if (u, v_neg) in forbidden_edges_set:  # Not in forbidden set
                continue
            
            neg_edges.append((u, v_neg))
            k += 1
        
        # If we couldn't find enough negatives, fill with random (rare for sparse graphs)
        while k < K:
            v_neg = int(rng.integers(0, n_nodes))
            if v_neg != u and v_neg != v_pos:
                neg_edges.append((u, v_neg))
                k += 1
    
    return np.array(neg_edges)


print("Per-source corruption negative sampling ready.")
print(f"  - sample_negatives_corrupt_tail: Samples K negatives per positive (same source, corrupted target)")

Per-source corruption negative sampling ready.
  - sample_negatives_corrupt_tail: Samples K negatives per positive (same source, corrupted target)


## 5.6 Ranking Metrics: MRR and Hits@K

With per-source corruption negatives, we evaluate the model's ability to **rank** the true target above corrupted candidates.

### Evaluation Protocol

For each test positive $(u \to v^+)$:
1. Gather the candidate set: $\{v^+, v^-_1, \ldots, v^-_K\}$ (1 positive + K negatives)
2. Compute predicted probability $P(y=1 | u, v)$ for all candidates
3. Rank candidates by probability (descending)
4. Record the rank of $v^+$

### Metrics

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **MRR** | $\frac{1}{n}\sum_i \frac{1}{\text{rank}_i}$ | Average reciprocal rank (higher = better) |
| **Hits@1** | $\frac{1}{n}\sum_i \mathbb{1}[\text{rank}_i = 1]$ | Fraction where true target is top-ranked |
| **Hits@3** | $\frac{1}{n}\sum_i \mathbb{1}[\text{rank}_i \leq 3]$ | Fraction where true target is in top 3 |
| **Hits@10** | $\frac{1}{n}\sum_i \mathbb{1}[\text{rank}_i \leq 10]$ | Fraction where true target is in top 10 |

### Why These Metrics Matter

- **MRR** is the primary metric in OGB link prediction leaderboards
- **Hits@K** measures practical utility: "Can I find the right answer in top K suggestions?"
- Unlike AUC, these metrics directly measure ranking quality for the recommendation task

In [17]:
def compute_ranking_metrics(pos_probs, neg_probs_matrix, K):
    """
    Compute MRR and Hits@K metrics for ranking evaluation.
    
    For each positive edge, we rank the true target against K negative candidates.
    
    Parameters:
    -----------
    pos_probs : array, shape (n_pos,)
        Predicted probability for each positive edge (the true target)
    neg_probs_matrix : array, shape (n_pos, K)
        Predicted probabilities for K negative candidates per positive.
        neg_probs_matrix[i, :] are the K negatives for positive i.
    K : int
        Number of negative candidates per positive
    
    Returns:
    --------
    dict with keys:
        'MRR': Mean Reciprocal Rank
        'Hits@1': Fraction with rank = 1
        'Hits@3': Fraction with rank <= 3  
        'Hits@10': Fraction with rank <= 10
        'ranks': array of all ranks (for debugging/analysis)
    """
    n_pos = len(pos_probs)
    ranks = []
    
    for i in range(n_pos):
        # Candidate probabilities: 1 positive + K negatives
        pos_prob = pos_probs[i]
        neg_probs = neg_probs_matrix[i, :]
        
        # Count how many negatives have higher probability than the positive
        # Rank 1 means the positive is top-ranked (no negatives beat it)
        n_better = np.sum(neg_probs > pos_prob)
        
        # Handle ties: use average rank (0.5 * n_ties) for fair tie-breaking
        n_ties = np.sum(neg_probs == pos_prob)
        
        # Rank = 1 + number of candidates with strictly higher probability + half of ties
        rank = 1 + n_better + 0.5 * n_ties
        ranks.append(rank)
    
    ranks = np.array(ranks)
    
    # Compute metrics
    mrr = np.mean(1.0 / ranks)
    hits_1 = np.mean(ranks == 1)
    hits_3 = np.mean(ranks <= 3)
    hits_10 = np.mean(ranks <= 10)
    
    return {
        'MRR': mrr,
        'Hits@1': hits_1,
        'Hits@3': hits_3,
        'Hits@10': hits_10,
        'ranks': ranks
    }


print("Ranking metrics (MRR, Hits@K) ready.")
print("  - compute_ranking_metrics: Computes MRR, Hits@1, Hits@3, Hits@10 from probability scores")
print("  - Uses fair tie-breaking: rank = 1 + n_better + 0.5 * n_ties")

Ranking metrics (MRR, Hits@K) ready.
  - compute_ranking_metrics: Computes MRR, Hits@1, Hits@3, Hits@10 from probability scores
  - Uses fair tie-breaking: rank = 1 + n_better + 0.5 * n_ties


## 6. HeaRT-Style Pipeline with Corruption Negatives

This pipeline implements  from the HeaRT paper

### Key Differences from Uniform Negative Sampling

| Component | Old Pipeline | New Pipeline (HeaRT-style) |
|-----------|-------------|---------------------------|
| **Negative sampling** | Uniform random $(u', v')$ | Per-source corruption $(u, v^-)$ |
| **Negatives per positive** | 1:1 balanced | K:1 ratio (default K=50) |
| **Primary metrics** | AUC, F1 | MRR, Hits@K |
| **Task framing** | Binary classification | Ranking / recommendation |

### Conformal Prediction Consistency

**Critical**: Calibration and test sets use the **same negative sampling scheme**:
- Both use K corruption negatives per positive
- Coverage guarantee still holds under exchangeability
- Prediction sets now answer: "Is the true target likely in the top-K?"

### Expected Outcomes

1. **Lower AUC** compared to uniform negatives (harder task)
2. **More meaningful metrics** (MRR, Hits@K reflect real recommendation quality)
3. **Better generalization** to real-world link prediction tasks

In [18]:
def run_directed_pipeline_heart(adj_path, name="dataset", alpha=0.10, n_runs=5, n_components=32, 
                                  msg_ratio=0.7, K_neg=50):
    """
    HeaRT-style directed graph pipeline with per-source corruption negatives and ranking metrics.
    
    Parameters:
    -----------
    adj_path : str
        Path to directed adjacency.tsv file
    name : str
        Dataset name for display
    alpha : float
        Target error rate (1 - coverage)
    n_runs : int
        Number of random runs for averaging
    n_components : int
        SVD embedding dimension
    msg_ratio : float
        Fraction of training edges used for message passing / SVD (default 0.7)
    K_neg : int
        Number of negative candidates per positive edge (default 50)
    
    Returns:
    --------
    results_df : DataFrame
        Results from all runs with both classification and ranking metrics
    """
    print(f"\n{'='*60}")
    print(f"Running HeaRT-Style Pipeline: {name.upper()}")
    print(f"Per-source corruption negatives (K={K_neg})")
    print(f"Averaging over {n_runs} runs")
    print(f"Message/Supervision split: {msg_ratio:.0%} / {1-msg_ratio:.0%}")
    print(f"{'='*60}")
    
    # Load directed graph
    adj = load_directed_graph_sparse(adj_path)
    n_nodes = adj.shape[0]
    print(f"Loaded directed graph: {n_nodes} nodes, {adj.nnz} edges")
    
    # Get all directed edges
    rows, cols = adj.nonzero()
    all_edges = np.column_stack([rows, cols])
    print(f"Total directed edges: {len(all_edges)}")
    
    # ============================================================
    # FIX: Build all_pos_set and out_neighbors_all from ALL observed edges
    # This ensures negatives are never observed positives in the dataset
    # ============================================================
    all_pos_set = set(map(tuple, all_edges))  # All observed positives
    out_neighbors_all = build_out_neighbors(all_edges, n_nodes)  # source -> all observed targets
    print(f"Built all_pos_set with {len(all_pos_set)} observed edges")
    
    run_results = []
    last_run_data = {}
    
    for seed in range(n_runs):
        rng = np.random.default_rng(seed)
        
        # Shuffle edges for exchangeability
        edges = all_edges.copy()
        rng.shuffle(edges)
        
        # Split: 10% cal, 10% test, 80% train
        n = len(edges)
        n_cal = int(n * 0.1)
        n_test = int(n * 0.1)
        
        E_cal = edges[:n_cal]
        E_test = edges[n_cal:n_cal + n_test]
        E_train_all = edges[n_cal + n_test:]
        
        # OGB-style: Split training edges into message (for SVD) and supervision (for classifier)
        n_train = len(E_train_all)
        n_msg = int(n_train * msg_ratio)
        
        train_idx = rng.permutation(n_train)
        E_msg = E_train_all[train_idx[:n_msg]]
        E_sup = E_train_all[train_idx[n_msg:]]
        
        # Build representation graph from MESSAGE edges only
        msg_rows = E_msg[:, 0]
        msg_cols = E_msg[:, 1]
        msg_data = np.ones(len(msg_rows))
        adj_msg = sp.csr_matrix((msg_data, (msg_rows, msg_cols)), shape=(n_nodes, n_nodes))
        adj_msg.data = np.ones_like(adj_msg.data)
        adj_msg.setdiag(0)
        adj_msg.eliminate_zeros()
        
        # Compute features only from message graph
        out_deg, in_deg, src_emb, tgt_emb = get_directed_node_features(adj_msg, n_components)
        
        # ============================================================
        # TRAINING DATA: Per-source corruption negatives
        # FIX: Use out_neighbors_all and all_pos_set for filtering
        # ============================================================
        train_neg = sample_negatives_corrupt_tail(
            E_sup, K_neg, rng, n_nodes, out_neighbors_all, forbidden_edges_set=all_pos_set
        )
        
        X_train_pos = compute_directed_pair_features(E_sup, out_deg, in_deg, src_emb, tgt_emb)
        X_train_neg = compute_directed_pair_features(train_neg, out_deg, in_deg, src_emb, tgt_emb)
        
        X_train = np.vstack([X_train_pos, X_train_neg])
        y_train = np.hstack([np.ones(len(E_sup)), np.zeros(len(train_neg))])
        
        # ============================================================
        # CALIBRATION DATA: Same corruption scheme
        # FIX: Use out_neighbors_all and all_pos_set for filtering
        # ============================================================
        cal_neg = sample_negatives_corrupt_tail(
            E_cal, K_neg, rng, n_nodes, out_neighbors_all, forbidden_edges_set=all_pos_set
        )
        
        X_cal_pos = compute_directed_pair_features(E_cal, out_deg, in_deg, src_emb, tgt_emb)
        X_cal_neg = compute_directed_pair_features(cal_neg, out_deg, in_deg, src_emb, tgt_emb)
        
        X_cal = np.vstack([X_cal_pos, X_cal_neg])
        y_cal = np.hstack([np.ones(len(E_cal)), np.zeros(len(cal_neg))])
        
        # ============================================================
        # TEST DATA: Same corruption scheme
        # FIX: Use out_neighbors_all and all_pos_set for filtering
        # ============================================================
        test_neg = sample_negatives_corrupt_tail(
            E_test, K_neg, rng, n_nodes, out_neighbors_all, forbidden_edges_set=all_pos_set
        )
        
        # ============================================================
        # SANITY CHECK: Verify negative sampling quality
        # ============================================================
        overlap = sum((u, v) in all_pos_set for (u, v) in map(tuple, test_neg))
        print(f"Run {seed+1}: test_neg overlap with observed positives: {overlap} (should be 0)")
        
        assert len(test_neg) == len(E_test) * K_neg, \
            f"Group shape mismatch: {len(test_neg)} != {len(E_test) * K_neg}"
        
        X_test_pos = compute_directed_pair_features(E_test, out_deg, in_deg, src_emb, tgt_emb)
        X_test_neg = compute_directed_pair_features(test_neg, out_deg, in_deg, src_emb, tgt_emb)
        
        X_test = np.vstack([X_test_pos, X_test_neg])
        y_test = np.hstack([np.ones(len(E_test)), np.zeros(len(test_neg))])
        
        # ============================================================
        # TRAIN CLASSIFIER
        # ============================================================
        clf = GradientBoostingClassifier(random_state=seed, n_estimators=100)
        clf.fit(X_train, y_train)
        
        # Predict probabilities
        p_cal = clf.predict_proba(X_cal)[:, 1]
        p_test = clf.predict_proba(X_test)[:, 1]
        
        # ============================================================
        # RANKING METRICS (MRR, Hits@K)
        # ============================================================
        # Get probabilities for positives and their K negatives
        n_test_pos = len(E_test)
        test_pos_probs = p_test[:n_test_pos]
        test_neg_probs = p_test[n_test_pos:].reshape(n_test_pos, K_neg)
        
        ranking_results = compute_ranking_metrics(test_pos_probs, test_neg_probs, K_neg)
        
        # ============================================================
        # CONFORMAL PREDICTION (on imbalanced data)
        # ============================================================
        n_cal_pos = len(E_cal)
        cal_pos_probs = p_cal[:n_cal_pos]
        cal_neg_probs = p_cal[n_cal_pos:]
        
        # Mondrian conformal quantiles
        scores_pos = 1 - cal_pos_probs
        scores_neg = cal_neg_probs
        
        q1 = conformal_quantile(scores_pos, alpha)
        q0 = conformal_quantile(scores_neg, alpha)
        
        t_lower = q0
        t_upper = 1 - q1
        
        # Generate prediction sets
        sets = []
        for p in p_test:
            s = set()
            if p <= t_lower:
                s.add(0)
            if p >= t_upper:
                s.add(1)
            if len(s) == 0:
                s = {0, 1}
            sets.append(s)
        
        # ============================================================
        # COVERAGE METRICS (including class-conditional)
        # ============================================================
        y_test_arr = np.array(y_test)
        set_sizes = np.array([len(s) for s in sets])
        
        # Overall coverage
        coverage = np.mean([y in s for y, s in zip(y_test_arr, sets)])
        
        # Class-conditional coverage (for Mondrian CP reporting)
        pos_mask = y_test_arr == 1
        neg_mask = y_test_arr == 0
        sets_arr = np.array(sets, dtype=object)
        
        cov_pos = np.mean([y in s for y, s in zip(y_test_arr[pos_mask], sets_arr[pos_mask])])
        cov_neg = np.mean([y in s for y, s in zip(y_test_arr[neg_mask], sets_arr[neg_mask])])
        
        avg_set_size = np.mean(set_sizes)
        abstention = np.mean([len(s) > 1 for s in sets])
        
        print(f"  Coverage(all)={coverage:.4f} | Coverage(pos)={cov_pos:.4f} | "
              f"Coverage(neg)={cov_neg:.4f} | AvgSetSize={avg_set_size:.3f}")
        
        # F1 on decided samples
        decided_idx = [i for i, s in enumerate(sets) if len(s) == 1]
        if decided_idx:
            y_dec = y_test[decided_idx]
            p_dec = [list(sets[i])[0] for i in decided_idx]
            _, _, f1_dec, _ = precision_recall_fscore_support(y_dec, p_dec, average='binary', zero_division=0)
        else:
            f1_dec = 0.0
        
        # AUC (on imbalanced test set)
        auc = roc_auc_score(y_test, p_test)
        
        run_results.append({
            "AUC": auc,
            "MRR": ranking_results['MRR'],
            "Hits@1": ranking_results['Hits@1'],
            "Hits@3": ranking_results['Hits@3'],
            "Hits@10": ranking_results['Hits@10'],
            "Coverage": coverage,
            "Coverage_pos": cov_pos,
            "Coverage_neg": cov_neg,
            "AvgSetSize": avg_set_size,
            "Abstention": abstention,
            "Decided F1": f1_dec
        })
        
        # Store last run data
        last_run_data = {
            'clf': clf,
            'y_test': y_test,
            'p_test': p_test,
            'X_test': X_test,
            'E_test': E_test,
            'test_neg': test_neg,
            't_lower': t_lower,
            't_upper': t_upper,
            'sets': sets,
            'auc': auc,
            'ranking_results': ranking_results,
            'out_deg': out_deg,
            'in_deg': in_deg
        }
        
        print(f"Run {seed+1}/{n_runs}: MRR={ranking_results['MRR']:.4f} | "
              f"H@1={ranking_results['Hits@1']:.4f} | H@10={ranking_results['Hits@10']:.4f} | "
              f"Cov={coverage:.4f}")
    
    # Aggregate results
    df = pd.DataFrame(run_results)
    means = df.mean()
    stds = df.std()
    
    print(f"\n{'='*60}")
    print(f"FINAL RESULTS - HeaRT-Style ({name.upper()})")
    print(f"{'='*60}")
    print(f"Splits: E_msg={n_msg}, E_sup={n_train-n_msg}, E_cal={n_cal}, E_test={n_test}")
    print(f"Negatives per positive: K={K_neg}")
    print(f"\n--- Ranking Metrics (Primary) ---")
    for col in ['MRR', 'Hits@1', 'Hits@3', 'Hits@10']:
        print(f"{col:15s}: {means[col]:.4f} +/- {stds[col]:.4f}")
    print(f"\n--- Conformal Prediction Metrics ---")
    for col in ['Coverage', 'Coverage_pos', 'Coverage_neg', 'AvgSetSize', 'Abstention', 'Decided F1']:
        print(f"{col:15s}: {means[col]:.4f} +/- {stds[col]:.4f}")
    print(f"\n--- Classification Metrics ---")
    print(f"{'AUC':15s}: {means['AUC']:.4f} +/- {stds['AUC']:.4f}")
    
    # ==========================================
    # FEATURE IMPORTANCE ANALYSIS (Last Run)
    # ==========================================
    print(f"\n{'='*60}")
    print(f"FEATURE IMPORTANCE (Last Run)")
    print(f"{'='*60}")
    
    clf = last_run_data['clf']
    feature_importance = clf.feature_importances_
    sorted_idx = np.argsort(feature_importance)[::-1]
    feat_names = FEATURE_NAMES_DIRECTED
    
    for i, idx in enumerate(sorted_idx):
        print(f"{i+1}. {feat_names[idx]}: {feature_importance[idx]:.4f}")
    
    # ==========================================
    # VISUALIZATION (Last Run)
    # ==========================================
    ranking_results = last_run_data['ranking_results']
    ranks = ranking_results['ranks']
    t_lower = last_run_data['t_lower']
    t_upper = last_run_data['t_upper']
    y_test = last_run_data['y_test']
    p_test = last_run_data['p_test']
    sets = last_run_data['sets']
    auc = last_run_data['auc']
    
    plt.figure(figsize=(18, 10))
    
    # Plot 1: Rank distribution histogram
    plt.subplot(2, 3, 1)
    plt.hist(ranks, bins=np.arange(1, K_neg+2) - 0.5, color='steelblue', alpha=0.7, edgecolor='black')
    plt.axvline(1, color='green', linestyle='--', linewidth=2, label='Perfect rank')
    plt.axvline(10, color='orange', linestyle='--', linewidth=2, label='Hits@10 threshold')
    plt.xlabel('Rank of True Target')
    plt.ylabel('Count')
    plt.title(f'Rank Distribution (MRR={ranking_results["MRR"]:.3f})')
    plt.xlim(0.5, min(K_neg, 50) + 0.5)
    plt.legend()
    plt.grid(alpha=0.3, axis='y')
    
    # Plot 2: Hits@K bar chart
    plt.subplot(2, 3, 2)
    hits_metrics = ['Hits@1', 'Hits@3', 'Hits@10']
    hits_values = [ranking_results[m] for m in hits_metrics]
    colors = ['#2ecc71', '#3498db', '#9b59b6']
    bars = plt.bar(hits_metrics, hits_values, color=colors, alpha=0.8)
    plt.ylabel('Fraction')
    plt.title('Hits@K Metrics')
    plt.ylim(0, 1)
    for bar, val in zip(bars, hits_values):
        plt.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.3f}', ha='center', fontsize=11)
    plt.grid(alpha=0.3, axis='y')
    
    # Plot 3: Probability distributions (positives vs negatives)
    plt.subplot(2, 3, 3)
    n_test_pos = len(last_run_data['E_test'])
    test_pos_probs = p_test[:n_test_pos]
    test_neg_probs = p_test[n_test_pos:]
    
    sns.histplot(test_neg_probs, label='Negatives', color='red', alpha=0.4, kde=True, bins=30)
    sns.histplot(test_pos_probs, label='Positives', color='blue', alpha=0.4, kde=True, bins=30)
    plt.axvline(t_lower, color='red', linestyle='--', linewidth=2, label=f't_lower={t_lower:.3f}')
    plt.axvline(t_upper, color='blue', linestyle='--', linewidth=2, label=f't_upper={t_upper:.3f}')
    plt.xlabel('Predicted Probability')
    plt.ylabel('Count')
    plt.title('Probability Distributions')
    plt.legend()
    plt.grid(alpha=0.3)
    
    # Plot 4: Feature Importance
    plt.subplot(2, 3, 4)
    colors = plt.cm.viridis(np.linspace(0, 0.8, len(feat_names)))
    sorted_importance = [feature_importance[i] for i in sorted_idx]
    sorted_names = [feat_names[i] for i in sorted_idx]
    bars = plt.barh(range(len(sorted_names)), sorted_importance[::-1], color=colors)
    plt.yticks(range(len(sorted_names)), sorted_names[::-1])
    plt.xlabel("Importance")
    plt.title(f"Feature Importance: {name.upper()}")
    for i, (bar, val) in enumerate(zip(bars, sorted_importance[::-1])):
        plt.text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)
    plt.grid(alpha=0.3, axis='x')
    
    # Plot 5: Prediction set sizes
    plt.subplot(2, 3, 5)
    set_sizes = [len(s) for s in sets]
    unique, counts = np.unique(set_sizes, return_counts=True)
    plt.bar(unique, counts, color=['green', 'orange'], alpha=0.7)
    plt.xlabel('Prediction Set Size')
    plt.ylabel('Count')
    plt.title('Conformal Prediction Sets')
    plt.xticks([1, 2], ['Decided', 'Abstain'])
    plt.grid(alpha=0.3, axis='y')
    
    # Plot 6: Metrics comparison summary
    plt.subplot(2, 3, 6)
    metric_names = ['MRR', 'Hits@1', 'Hits@10', 'Coverage', 'AUC']
    metric_values = [ranking_results['MRR'], ranking_results['Hits@1'], 
                     ranking_results['Hits@10'], 
                     np.mean([y in s for y, s in zip(y_test, sets)]), auc]
    colors = ['#e74c3c', '#2ecc71', '#9b59b6', '#f39c12', '#3498db']
    bars = plt.bar(metric_names, metric_values, color=colors, alpha=0.8)
    plt.ylabel('Value')
    plt.title('Metrics Summary (Last Run)')
    plt.ylim(0, 1)
    for bar, val in zip(bars, metric_values):
        plt.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.3f}', ha='center', fontsize=10)
    plt.grid(alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    return df, {
        'feature_importance': dict(zip(feat_names, feature_importance.tolist())),
        'last_run_ranking': ranking_results
    }


print("HeaRT-style pipeline with corruption negatives ready.")
print("  - Uses per-source corruption negatives (K negatives per positive)")
print("  - Filters negatives against ALL observed edges (all_pos_set)")
print("  - Reports MRR, Hits@1, Hits@3, Hits@10 as primary metrics")
print("  - Reports class-conditional coverage (cov_pos, cov_neg)")
print("  - Includes sanity checks for negative sampling quality")

HeaRT-style pipeline with corruption negatives ready.
  - Uses per-source corruption negatives (K negatives per positive)
  - Filters negatives against ALL observed edges (all_pos_set)
  - Reports MRR, Hits@1, Hits@3, Hits@10 as primary metrics
  - Reports class-conditional coverage (cov_pos, cov_neg)
  - Includes sanity checks for negative sampling quality


In [19]:
FEATURE_NAMES_DIRECTED = ['out_deg_u', 'in_deg_v', 'in_deg_u', 'out_deg_v', 'sim_forward', 'sim_reverse']

def run_directed_pipeline(adj_path, name="dataset", alpha=0.10, n_runs=10, n_components=32, msg_ratio=0.7):
    """
    
    Parameters:
    -----------
    adj_path : str
        Path to directed adjacency.tsv file
    name : str
        Dataset name for display
    alpha : float
        Target error rate (1 - coverage)
    n_runs : int
        Number of random runs for averaging
    n_components : int
        SVD embedding dimension
    msg_ratio : float
        Fraction of training edges used for message passing / SVD (default 0.7)
        Remaining (1 - msg_ratio) used for classifier supervision
    
    Returns:
    --------
    results_df : DataFrame
        Results from all runs
    """
    print(f"\n{'='*50}")
    print(f"Running Directed Graph Pipeline: {name.upper()}")
    print(f"Averaging over {n_runs} runs")
    print(f"Message/Supervision split: {msg_ratio:.0%} / {1-msg_ratio:.0%}")
    print(f"{'='*50}")
    
    # Load directed graph
    adj = load_directed_graph_sparse(adj_path)
    n_nodes = adj.shape[0]
    print(f"Loaded directed graph: {n_nodes} nodes, {adj.nnz} edges")
    
    # Get all directed edges
    rows, cols = adj.nonzero()
    all_edges = np.column_stack([rows, cols])
    print(f"Total directed edges: {len(all_edges)}")
    
    run_results = []
    
    # Store last run data for visualization and error analysis
    last_run_data = {}
    
    for seed in range(n_runs):
        rng = np.random.default_rng(seed)
        
        # Shuffle edges for exchangeability
        edges = all_edges.copy()
        rng.shuffle(edges)
        
        # Split: 10% cal, 10% test, 80% train
        n = len(edges)
        n_cal = int(n * 0.1)
        n_test = int(n * 0.1)
        
        E_cal = edges[:n_cal]
        E_test = edges[n_cal:n_cal + n_test]
        E_train_all = edges[n_cal + n_test:]
        
        # OGB-style: Split training edges into message (for SVD) and supervision (for classifier)
        n_train = len(E_train_all)
        n_msg = int(n_train * msg_ratio)
        
        # Shuffle training edges before splitting
        train_idx = rng.permutation(n_train)
        E_msg = E_train_all[train_idx[:n_msg]]    # For building G_rep (SVD)
        E_sup = E_train_all[train_idx[n_msg:]]    # For classifier supervision
        
        # ============================================================
        # REALISTIC NEGATIVE SAMPLING (No oracle access to test edges)
        # ============================================================
        # known_edges_set: edges we "know" before evaluation (training edges only)
        # This prevents data leakage from using test positives to filter negatives
        known_edges_set = set(map(tuple, np.vstack([E_msg, E_sup])))
        
        # Build representation graph from MESSAGE edges only (not supervision edges)
        # For directed graphs, we do NOT add reverse edges
        msg_rows = E_msg[:, 0]
        msg_cols = E_msg[:, 1]
        msg_data = np.ones(len(msg_rows))
        adj_msg = sp.csr_matrix((msg_data, (msg_rows, msg_cols)), shape=(n_nodes, n_nodes))
        adj_msg.data = np.ones_like(adj_msg.data)
        adj_msg.setdiag(0)
        adj_msg.eliminate_zeros()
        
        # Compute features only from message graph
        out_deg, in_deg, src_emb, tgt_emb = get_directed_node_features(adj_msg, n_components)
        
        def make_dataset(pos_edges, exclude_set):
            """
            Create balanced dataset with positive and negative edges.
            
            Parameters:
            -----------
            pos_edges : array
                Positive edge pairs for this split
            exclude_set : set
                Set of edges to exclude when sampling negatives.
                This should NOT include oracle knowledge of other splits' positives.
            """
            neg_edges = set()
            n_needed = len(pos_edges)
            
            # Sample non-edges, excluding only edges we legitimately know about
            while len(neg_edges) < n_needed:
                u, v = rng.integers(0, n_nodes, 2)
                if u != v and (u, v) not in exclude_set:
                    neg_edges.add((u, v))
            
            neg_edges = np.array(list(neg_edges))
            
            # Compute features
            X_pos = compute_directed_pair_features(pos_edges, out_deg, in_deg, src_emb, tgt_emb)
            X_neg = compute_directed_pair_features(neg_edges, out_deg, in_deg, src_emb, tgt_emb)
            
            X = np.vstack([X_pos, X_neg])
            y = np.hstack([np.ones(len(pos_edges)), np.zeros(len(neg_edges))])
            
            # Return pairs for error analysis
            all_pairs = np.vstack([pos_edges, neg_edges])
            
            return X, y, all_pairs
        
        # Train: exclude known training edges (E_msg + E_sup)
        X_train, y_train, _ = make_dataset(E_sup, exclude_set=known_edges_set)
        
        # Cal: exclude known edges + calibration positives (we know them when labeling cal)
        cal_exclude = known_edges_set | set(map(tuple, E_cal))
        X_cal, y_cal, _ = make_dataset(E_cal, exclude_set=cal_exclude)
        
        # Test: exclude only known train edges (DO NOT exclude E_test positives - no oracle!)
        X_test, y_test, test_pairs = make_dataset(E_test, exclude_set=known_edges_set)
        
        # Train classifier
        clf = GradientBoostingClassifier(random_state=seed, n_estimators=100)
        clf.fit(X_train, y_train)
        
        # Predict probabilities
        p_cal = clf.predict_proba(X_cal)[:, 1]
        p_test = clf.predict_proba(X_test)[:, 1]
        
        # Mondrian conformal quantiles
        scores_pos = 1 - p_cal[y_cal == 1]  # Score for y=1
        scores_neg = p_cal[y_cal == 0]      # Score for y=0
        
        q1 = conformal_quantile(scores_pos, alpha)
        q0 = conformal_quantile(scores_neg, alpha)
        
        # Thresholds
        t_lower = q0
        t_upper = 1 - q1
        
        # Generate prediction sets
        sets = []
        for p in p_test:
            s = set()
            if p <= t_lower:
                s.add(0)
            if p >= t_upper:
                s.add(1)
            if len(s) == 0:  # Abstain
                s = {0, 1}
            sets.append(s)
        
        # Compute metrics
        coverage = np.mean([y in s for y, s in zip(y_test, sets)])
        abstention = np.mean([len(s) > 1 for s in sets])
        
        # F1 on decided samples
        decided_idx = [i for i, s in enumerate(sets) if len(s) == 1]
        if decided_idx:
            y_dec = y_test[decided_idx]
            p_dec = [list(sets[i])[0] for i in decided_idx]
            _, _, f1_dec, _ = precision_recall_fscore_support(y_dec, p_dec, average='binary', zero_division=0)
        else:
            f1_dec = 0.0
        
        # Baseline metrics
        auc = roc_auc_score(y_test, p_test)
        y_pred_std = (p_test >= 0.5).astype(int)
        _, _, base_f1, _ = precision_recall_fscore_support(y_test, y_pred_std, average='binary', zero_division=0)
        
        run_results.append({
            "AUC": auc,
            "Base F1": base_f1,
            "Coverage": coverage,
            "Abstention": abstention,
            "Decided F1": f1_dec
        })
        
        # Store last run data for visualization and error analysis
        last_run_data = {
            'clf': clf,
            'y_test': y_test,
            'p_test': p_test,
            'y_pred_std': y_pred_std,
            'X_test': X_test,
            'test_pairs': test_pairs,
            't_lower': t_lower,
            't_upper': t_upper,
            'sets': sets,
            'auc': auc
        }
        
        print(f"Run {seed+1}/{n_runs}: AUC={auc:.4f} | Cov={coverage:.4f} | Abs={abstention:.4f}")
    
    # Aggregate results
    df = pd.DataFrame(run_results)
    means = df.mean()
    stds = df.std()
    
    print(f"\n{'='*50}")
    print(f"FINAL RESULTS ({name.upper()})")
    print(f"{'='*50}")
    print(f"Splits: E_msg={n_msg}, E_sup={n_train-n_msg}, E_cal={n_cal}, E_test={n_test}")
    for col in df.columns:
        print(f"{col:15s}: {means[col]:.4f} ± {stds[col]:.4f}")
    
    # ==========================================
    # FEATURE IMPORTANCE ANALYSIS (Last Run)
    # ==========================================
    print(f"\n{'='*50}")
    print(f"FEATURE IMPORTANCE (Last Run)")
    print(f"{'='*50}")
    
    clf = last_run_data['clf']
    feature_importance = clf.feature_importances_
    sorted_idx = np.argsort(feature_importance)[::-1]
    
    for i, idx in enumerate(sorted_idx):
        print(f"{i+1}. {FEATURE_NAMES_DIRECTED[idx]}: {feature_importance[idx]:.4f}")
    
    # ==========================================
    # ERROR ANALYSIS (Last Run)
    # ==========================================
    print(f"\n{'='*50}")
    print(f"ERROR ANALYSIS (Last Run)")
    print(f"{'='*50}")
    
    y_test = last_run_data['y_test']
    p_test = last_run_data['p_test']
    y_pred_std = last_run_data['y_pred_std']
    X_test = last_run_data['X_test']
    test_pairs = last_run_data['test_pairs']
    
    # Identify false positives and false negatives
    fp_indices = np.where((y_test == 0) & (y_pred_std == 1))[0]
    fn_indices = np.where((y_test == 1) & (y_pred_std == 0))[0]
    
    # Sort by confidence (how wrong the model was)
    fp_sorted = fp_indices[np.argsort(p_test[fp_indices])[::-1]] if len(fp_indices) > 0 else []
    fn_sorted = fn_indices[np.argsort(p_test[fn_indices])] if len(fn_indices) > 0 else []
    
    print(f"\nTotal False Positives: {len(fp_indices)}")
    print(f"Total False Negatives: {len(fn_indices)}")
    
    feat_names = FEATURE_NAMES_DIRECTED
    
    print(f"\n--- Top 10 False Positives (predicted edge, actual non-edge) ---")
    print(f"{'Rank':<5} {'Pair (u->v)':<20} {'Prob':<8} {' | '.join([f'{f[:8]:<10}' for f in feat_names])}")
    print("-" * (35 + 11 * len(feat_names)))
    for rank, idx in enumerate(fp_sorted[:10], 1):
        pair = test_pairs[idx]
        prob = p_test[idx]
        feats = X_test[idx]
        feat_str = ' | '.join([f'{feats[i]:<10.2f}' for i in range(len(feat_names))])
        print(f"{rank:<5} ({pair[0]}->{pair[1]})".ljust(25) + f" {prob:<8.4f} {feat_str}")
    
    print(f"\n--- Top 10 False Negatives (predicted non-edge, actual edge) ---")
    print(f"{'Rank':<5} {'Pair (u->v)':<20} {'Prob':<8} {' | '.join([f'{f[:8]:<10}' for f in feat_names])}")
    print("-" * (35 + 11 * len(feat_names)))
    for rank, idx in enumerate(fn_sorted[:10], 1):
        pair = test_pairs[idx]
        prob = p_test[idx]
        feats = X_test[idx]
        feat_str = ' | '.join([f'{feats[i]:<10.2f}' for i in range(len(feat_names))])
        print(f"{rank:<5} ({pair[0]}->{pair[1]})".ljust(25) + f" {prob:<8.4f} {feat_str}")
    
    # ==========================================
    # VISUALIZATION (Last Run)
    # ==========================================
    t_lower = last_run_data['t_lower']
    t_upper = last_run_data['t_upper']
    sets = last_run_data['sets']
    auc = last_run_data['auc']
    
    plt.figure(figsize=(16, 10))
    
    # Plot 1: ROC curve
    plt.subplot(2, 3, 1)
    fpr, tpr, _ = roc_curve(y_test, p_test)
    plt.plot(fpr, tpr, label=f"AUC={auc:.3f}", color='blue', linewidth=2)
    plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve: {name.upper()}')
    plt.legend()
    plt.grid(alpha=0.3)
    
    # Plot 2: Probability distributions
    plt.subplot(2, 3, 2)
    sns.histplot(p_test[y_test == 0], label='No Edge (y=0)', color='red', alpha=0.4, kde=True, bins=30)
    sns.histplot(p_test[y_test == 1], label='Edge (y=1)', color='blue', alpha=0.4, kde=True, bins=30)
    plt.axvline(t_lower, color='red', linestyle='--', linewidth=2, label=f't_lower={t_lower:.3f}')
    plt.axvline(t_upper, color='blue', linestyle='--', linewidth=2, label=f't_upper={t_upper:.3f}')
    plt.xlabel('Predicted Probability')
    plt.ylabel('Count')
    plt.title('Conformal Thresholds')
    plt.legend()
    plt.grid(alpha=0.3)
    
    # Plot 3: Prediction set sizes
    plt.subplot(2, 3, 3)
    set_sizes = [len(s) for s in sets]
    unique, counts = np.unique(set_sizes, return_counts=True)
    plt.bar(unique, counts, color=['green', 'orange'], alpha=0.7)
    plt.xlabel('Prediction Set Size')
    plt.ylabel('Count')
    plt.title('Prediction Set Size Distribution')
    plt.xticks([1, 2], ['Decided', 'Abstain'])
    plt.grid(alpha=0.3, axis='y')
    
    # Plot 4: Feature Importance
    plt.subplot(2, 3, 4)
    colors = plt.cm.viridis(np.linspace(0, 0.8, len(feat_names)))
    sorted_importance = [feature_importance[i] for i in sorted_idx]
    sorted_names = [feat_names[i] for i in sorted_idx]
    bars = plt.barh(range(len(sorted_names)), sorted_importance[::-1], color=colors)
    plt.yticks(range(len(sorted_names)), sorted_names[::-1])
    plt.xlabel("Importance")
    plt.title(f"Feature Importance: {name.upper()}")
    for i, (bar, val) in enumerate(zip(bars, sorted_importance[::-1])):
        plt.text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)
    plt.grid(alpha=0.3, axis='x')
    
    # Plot 5: Error Distribution
    plt.subplot(2, 3, 5)
    if len(fp_indices) > 0:
        plt.hist(p_test[fp_indices], bins=20, alpha=0.5, label=f'False Positives (n={len(fp_indices)})', color='orange')
    if len(fn_indices) > 0:
        plt.hist(p_test[fn_indices], bins=20, alpha=0.5, label=f'False Negatives (n={len(fn_indices)})', color='purple')
    plt.axvline(0.5, color='black', linestyle='--', linewidth=2, label='Decision Threshold')
    plt.xlabel("Predicted Probability")
    plt.ylabel("Count")
    plt.title("Error Distribution")
    plt.legend()
    plt.grid(alpha=0.3)
    
    # Plot 6: Error counts by type
    plt.subplot(2, 3, 6)
    error_types = ['False Positives', 'False Negatives']
    error_counts = [len(fp_indices), len(fn_indices)]
    bars = plt.bar(error_types, error_counts, color=['orange', 'purple'], alpha=0.7)
    plt.ylabel("Count")
    plt.title("Error Breakdown")
    for bar, count in zip(bars, error_counts):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(count), ha='center', fontsize=11)
    plt.grid(alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # Return results with feature importance
    return df, {
        'feature_importance': dict(zip(feat_names, feature_importance.tolist())),
        'n_false_positives': len(fp_indices),
        'n_false_negatives': len(fn_indices)
    }

print("Directed pipeline ready.")

Directed pipeline ready.


## 6. Run Experiments on Wikivitals

### 6.1 About Wikivitals Dataset

The **Wikivitals** dataset contains:
- **Nodes**: 10,010 vital Wikipedia articles (level 4)
- **Directed edges**: ~825,000 hyperlinks between articles
- **Nature**: Information flow in knowledge graph

### 6.2 Experimental Setup

- **Confidence level**: $1 - \alpha = 0.90$ (90% coverage)
- **Embedding dimension**: $k = 32$
- **Runs**: 10 random splits for statistical significance
- **Classifier**: Gradient Boosting (100 trees)

In [ ]:
# Run HeaRT-Style Pipeline with K=50
results, analysis = run_directed_pipeline_heart(
    adj_path="wikivitals/adjacency.tsv",
    name="Wikivitals",
    alpha=0.10,
    n_runs=5,
    n_components=32,
    msg_ratio=0.7,
    K_neg=50
)

print("\n--- Analysis Summary ---")
print(f"Feature Importance: {analysis['feature_importance']}")
print(f"Ranking Results: {analysis['last_run_ranking']}")


Running HeaRT-Style Pipeline: WIKIVITALS
Per-source corruption negatives (K=50)
Averaging over 5 runs
Message/Supervision split: 70% / 30%
Loaded directed graph: 10011 nodes, 823619 edges
Total directed edges: 823619
Built all_pos_set with 823619 observed edges
Computing SVD Embeddings (Rank 32) for directed graph...
Run 1: test_neg overlap with observed positives: 0 (should be 0)
